EDA draft

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
DATA_FOLDER: str = "./data/"
UNICEF_DATA_CSV: str = DATA_FOLDER + "unicef_malawi.csv"
OUTPUT_FOLDER: str = "./output/"

In [18]:
unicef_df = pd.read_csv(UNICEF_DATA_CSV)

# Make target binary
Use map to transform target column "FCF26" to a binary column "FCF26_binary", where 0 = not depressed, and 1 = depressed

In [19]:
unicef_df = pd.read_csv(UNICEF_DATA_CSV)
print(f"FCF26 {unicef_df['FCF26'].isnull().mean() * 100:2f} % nan values")
unicef_df = unicef_df.dropna(subset=["FCF26"])

FCF26 0.782556 % nan values


In [20]:
# depression_map maps values in FCF26 to binary; where 0 = not depressed, and 1 = depressed.
depression_map: dict[str: int] = {
    "NEVER": 0,
    "A FEW TIMES A YEAR": 0,
    "MONTHLY": 0,
    "WEEKLY": 1,
    "DAILY": 1,
}

def map_target_to_binary(
    *,
    df: pd.DataFrame,
    value_to_binary: dict[str, int],
) -> pd.DataFrame:
    df["FCF26_binary"] = df["FCF26"].map(value_to_binary)
    return df

unicef_df = unicef_df[unicef_df["FCF26"] != "NO RESPONSE"]
unicef_df = map_target_to_binary(df=unicef_df, value_to_binary=depression_map)
print(f"FCF26_binary {unicef_df['FCF26_binary'].isnull().mean() * 100:2f} % nan values")
print(unicef_df['FCF26'].unique())

FCF26_binary 0.000000 % nan values
<StringArray>
['NEVER', 'DAILY', 'A FEW TIMES A YEAR', 'WEEKLY', 'MONTHLY']
Length: 5, dtype: str


# Rank correlations to target
We use Cramér's V to measure the association between each feature and the target, removing features with Cramér's V < 0.05, corresponding to less than 0.25% of variance explained individually. While individual features explain modest variance; reflecting the multifactorial and self-reported nature of the data, the combined predictive power of retained features is expected to be substantially higher, as the model learns from many weak signals jointly.

In [32]:
from scipy.stats import chi2_contingency
import numpy as np

def perform_cramers_v(
    *,
    df: pd.DataFrame,
    min_score: float = 0.05,
    target_column_name: str,
    ignore_columns: list,
) -> pd.DataFrame:
    """Performs cramers v test on the columns in the provided 'df' 
    against the 'target_column_name' column and returns a pd.DataFrame
    containing the signficiant features that meet the 'min_score'
    threshold. Ignores the 'ignore_columns'.
    """
    results = []
    df_without_target = df.drop(columns=ignore_columns)
    for col in df_without_target.columns:
        contingency_table = pd.crosstab(df[col], df[target_column_name])
        chi2, p, _, _ = chi2_contingency(contingency_table)
        n = contingency_table.sum().sum()
        min_dim = min(contingency_table.shape) - 1
        cramers_v = np.sqrt(chi2 / (n * min_dim))
        results.append({"feature_name": col, "p_value": p, "cramers_v": cramers_v})
    results_df = pd.DataFrame(results).sort_values("cramers_v", ascending=False)
    return results_df[results_df["cramers_v"] > min_score]

significant_features_df = perform_cramers_v(
    df=unicef_df,
    min_score=0.05,
    target_column_name="FCF26_binary",
    ignore_columns=["FCF26_binary", "FCF26"]

)
print(f"{len(significant_features_df)} features retained")
significant_features_df

29 features retained


,feature_name,p_value,cramers_v
28,wscore,5.303854e-01,0.998658
0,HH1,7.369146e-32,0.366855
10,CL3,9.147104e-03,0.129048
58,LS2,5.400523e-30,0.114343
57,LS1,4.425543e-28,0.104064
27,ethnicity,1.252945e-25,0.102337
80,WS4,2.086852e-02,0.100284
59,LS3,4.184646e-23,0.091711
25,HH7,9.319933e-23,0.088219
44,VT22B,1.747649e-19,0.084207


We remove w-score from our feature set as it represents a measure of child happiness; arguably an alternative target variable rather than a predictor, and its inclusion would risk circular reasoning. We additionally remove ethnicity to prevent the introduction of racial bias into a model that will inform resource allocation decisions. While ethnicity shows a weak but non-negligible association with depression (Cramér's V = 0.1, accounting for ~1% of variance), this association likely reflects proxy effects of socioeconomic status, cultural differences in self-reporting, or experiences of discrimination; factors better captured by other features in the dataset. We note that any future reintroduction of ethnicity would require significant ethical justification. This leaves us with 27 features.

In [22]:
def remove_features_from_df(
    *,
    df: pd.DataFrame,
    features_to_remove: list[str],
) -> pd.DataFrame:
    """Removes the features in 'features_to_remove' from the provided 'df' and returns the df.
    """
    return df[~df["feature_name"].isin(features_to_remove)] # removes features not in features_to_remove.

significant_features_df = remove_features_from_df(
    df=significant_features_df,
    features_to_remove = [
        "ethnicity", # Remove to prevent racial bias.
        "wscore", # Remove because wscore is a direct measure of happiness. Could be considered alternate target.
    ]
)
significant_features: list[str] = significant_features_df["feature_name"].to_list()
significant_features

['HH1',
 'CL3',
 'LS2',
 'LS1',
 'WS4',
 'LS3',
 'HH7',
 'VT22B',
 'MA2',
 'HC4',
 'VT22A',
 'CL13',
 'LS4',
 'WS11',
 'VT22C',
 'HH2',
 'WS7',
 'HC12',
 'MSTATUS',
 'FCD2H',
 'WS1',
 'FCD2J',
 'VT20',
 'HC5',
 'HC8',
 'VT22D',
 'WB4']

We copy our dataframe, keeping only the significant features identified, and the target "FCF26_binary"

In [23]:
unicef_df_filtered = unicef_df[significant_features + ["FCF26_binary"]].copy()

We further look at the percentage of nan values in a column. "CL3" has 68% values nan, "MA2" 23%, "CL13" 15%, "FCD2H" and "FCD2J" 14%, "WS4" 13%.

In [24]:
unicef_df_filtered.isnull().mean().sort_values(ascending=False) * 100

CL3             68.157410
MA2             22.468549
CL13            15.058300
FCD2H           13.769561
FCD2J           13.769561
WS4             13.355324
VT22A            2.140227
LS1              2.140227
VT22B            2.140227
VT22C            2.140227
LS2              2.140227
LS3              2.140227
VT22D            2.140227
WB4              2.140227
VT20             2.140227
LS4              2.140227
HC4              0.000000
HH7              0.000000
HH1              0.000000
WS11             0.000000
MSTATUS          0.000000
HC12             0.000000
WS7              0.000000
HH2              0.000000
HC5              0.000000
WS1              0.000000
HC8              0.000000
FCF26_binary     0.000000
dtype: float64

In [33]:
def identify_columns_of_type_object(df: pd.DataFrame):
    """Identifies the categorical columns that are object data types.
    These columns will need converting into numerical categories such that a model can understand them.
    """
    return df.select_dtypes(include=["object"]).columns.tolist()

categorical_cols = identify_columns_of_type_object(unicef_df_filtered)
print(f"{len(categorical_cols)} categorical object columns found: {categorical_cols}")

18 categorical object columns found: ['LS2', 'LS1', 'WS4', 'LS3', 'VT22B', 'MA2', 'VT22A', 'CL13', 'LS4', 'VT22C', 'WS7', 'HC12', 'MSTATUS', 'FCD2H', 'FCD2J', 'VT20', 'HC8', 'VT22D']


C:\Users\Ed\AppData\Local\Temp\ipykernel_16408\965583777.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  return df.select_dtypes(include=["object"]).columns.tolist()


We now need to transform categories of features into numerical representations, either one-hot encodings, binary, or ordinal.

In [34]:
for col in categorical_cols:
    print(f"\n{col}: {unicef_df_filtered[col].unique()}")


LS2: <StringArray>
['10', '1', '5', '6', '4', '8', nan, '0', '7', '3', '9', 'NO RESPONSE', '2']
Length: 13, dtype: str

LS1: <StringArray>
[         'SOMEWHAT UNHAPPY',            'SOMEWHAT HAPPY',
                'VERY HAPPY', 'NEITHER HAPPY NOR UNHAPPY',
              'VERY UNHAPPY',                         nan,
               'NO RESPONSE']
Length: 7, dtype: str

WS4: <StringArray>
[                   '5.0',                   '30.0',                    '6.0',
                    '8.0',                   '40.0',                      nan,
                   '10.0',                    '7.0',                   '60.0',
                   '15.0',                   '28.0',                   '45.0',
                   '20.0',                   '50.0',                   '70.0',
                   '90.0',                    '2.0',                   '25.0',
                  '120.0',                     'DK',                   '35.0',
                   '26.0',                   '16.0',      

# Full Pipeline

In [ ]:
class FeatureNames:
    """Stores feature names used throughout.
    Does not containt all 87 features in the dataset.
    """
    CL3 = "CL3"
    CL13 = "CL13"
    FCD2H = "FCD2H"
    FCD2J = "FCD2J"
    LS1 = "LS1"
    LS2 = "LS2"
    LS3 = "LS3"
    LS4 = "LS4"
    FCF26 = "FCF26"
    FCF26_BINARY = "FCF26_binary"
    ETHNICITY = "ethnicity"
    WSCORE = "wscore"
    HH7 = "HH7"
    HC4 = "HC4"
    HC5 = "HC5"
    HC8 = "HC8"
    HC12 = "HC12"
    MA2 = "MA2"
    MSTATUS = "MSTATUS"
    VT20 = "VT20"
    VT22A = "VT22A"
    VT22B = "VT22B"
    VT22C = "VT22C"
    VT22D = "VT22D"
    WS11 = "WS11"
    WS1 = "WS1"
    WS4 = "WS4"
    WS7 = "WS7"

# feature_config contains a map of feature name mapped to a dictionary of rules
# as to how the feature will be transformed. "map" is used to map non-numeric
# values to numeric ones. "numeric": True means the feature's values will be
# coerced into numbers; i.e "30" will become 30.
feature_config = {
    # LS1: To Mother: mother's happiness level, 4 categories
    FeatureNames.LS1: {
        "map": {
            "VERY UNHAPPY": 0,
            "SOMEWHAT UNHAPPY": 1,
            "NEITHER HAPPY NOR UNHAPPY": 2,
            "SOMEWHAT HAPPY": 3,
            "VERY HAPPY": 4,
            "NO RESPONSE": None,
        },
    },
    # LS2 To Mother: mother's happiness level 0-10
    FeatureNames.LS2: {
        "map": {
            "NO RESPONSE": None,
        },
        "numeric": True,
    },
    # LS3: To Mother: Compared to this time last year, would you say that your
    #      life has improved, stayed more or less the same, or worsened,
    #      overall?
    FeatureNames.LS3: {
        "map": {
            "WORSENED": 0,
            "MORE OR LESS THE SAME": 1,
            "IMPROVED": 2,
            "NO RESPONSE": None,
        },
    },
    # LS4: To Mother: And in one year from now, do you expect that your life
    #      will be better, will be more or less the same, or will be worse,
    #      overall?
    FeatureNames.LS4: {
        "map": {
            "WORSE": 0,
            "MORE OR LESS THE SAME": 1,
            "BETTER": 2,
            "NO RESPONSE": None,
        },
    },
    # WS4: How long does it take for members of your household to go there,
    #      get water, and come back?
    FeatureNames.WS4: {
        "map": {"DK": None, "MEMBERS DO NOT COLLECT": None, "NO RESPONSE": None}, "numeric": True,
    },
    # WS7: In the last month, has there been any time when your household did
    #      not have sufficient quantities of drinking water?
    FeatureNames.WS7: {
        "map": {
            "NO, ALWAYS SUFFICIENT": 0,
            "YES, AT LEAST ONCE": 1,
            "DK": None,
            "NO RESPONSE": None,
        },
    },
    # VT20: To Mother: How safe do you feel walking alone in your neighbourhood
    #       after dark?
    FeatureNames.VT20: {
        "map": {
            "VERY UNSAFE": 0,
            "UNSAFE": 1,
            "NEVER WALK ALONE AFTER DARK": 2,
            "SAFE": 3,
            "VERY SAFE": 4,
            "NO RESPONSE": None,
        },
    },
    # VT22: In the past 12 months, have you personally felt discriminated
    #       against or harassed on the basis of the following grounds?
    # VT22A: A) Ethnic or immigration origin?
    FeatureNames.VT22A: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22B: B) Sex?
    FeatureNames.VT22B: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22C: C) Sexual orientation?
    FeatureNames.VT22C: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22D: D) Age?
    FeatureNames.VT22D: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # MA2: How old is your (husband/partner)?
    FeatureNames.MA2: {"map": {"DK": None, "NO RESPONSE": None}, "numeric": True},
    # CL3: Since last (day of the week) about how many hours did (name) engage
    #      in (this activity/these activities), in total?
    #      Map np.nan values to 0 because if all answers to question CL2 are no,
    #      then question CL3 is skipped, implying 0 child labour hours worked.
    FeatureNames.CL3: {"map": {np.nan: 0}, "numeric": True},
    # CL13: Since last (day of the week), about how many hours did (name)
    #       engage in (this activity/these activities), in total?
    #       Again map np.nan values to 0. If all answers to question CL2 are
    #       no then question CL13 is skipped. There are 2 "NO RESPONSE" values.
    #       We map them to 0. Alternatively we could drop the two rows
    #       corresponding to "NO RESPONSE".
    FeatureNames.CL13: {
        "map": {
            "NO RESPONSE": 0,
            np.nan: 0
        },
        "numeric": True
    },
    # HC12: Does any member of your household have a mobile telephone?
    FeatureNames.HC12: {"map": {"NO": 0, "YES": 1, "NO RESPONSE": None}},
    # MSTATUS: marital status of mother
    FeatureNames.MSTATUS: {
        "map": {
            "Never married/in union": 0,
            "Formerly married/in union": 1,
            "Currently married/in union": 2,
        },
    },
    #FCD: Question to child: Adults use certain ways to teach children the 
    # right behaviour or to address a behaviour problem. I will read various
    # methods that are used. Please tell me if you or any other adult in your
    # household has used this method with (name) in the past month.

    # FCD2H: Called (him/her) dumb, lazy or another name like that.
    FeatureNames.FCD2H: {"map": {"NO": 0, "YES": 1, "NO RESPONSE": None}},
    # FCD2J: Hit or slapped (him/her) on the hand, arm, or leg.
    FeatureNames.FCD2J: {"map": {"NO": 0, "YES": 1, "NO RESPONSE": None}},
    # HC8: Does your household have electricity?
    FeatureNames.HC8: {
        "map": {
            "NO": 0,
            "YES, OFF-GRID (GENERATOR/ISOLATED SYSTEM)": 1,
            "YES, INTERCONNECTED GRID": 2,
            "NO RESPONSE": None,
        },
    },
    
}

def apply_feature_config(*, df: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = df.copy()
    for col, rules in config.items():
        if col not in df.columns:
            continue
        if value_map := rules.get("map"):
            df[col] = df[col].map(value_map).fillna(df[col])
        if rules.get("numeric"):
            df[col] = pd.to_numeric(df[col], errors="coerce")       
    return df

In [90]:
"""
 ====================================
  Feature Extraction and Engineering
 ====================================
  Section loads data, identifies the top features, manually removes features on further
  inspection. Then we convert all features to numerical representations, either binary,
  one-hot encodings (where a logical order does not make sense), and ordinal values in
  a logical order of significance. These steps are taken to transform the features
  into numerical representations that our model can "understand".
"""
    
# 1) Read unicef data into a Pandas DataFrame.
unicef_df = pd.read_csv(UNICEF_DATA_CSV)

# 2) Drop rows corresponding to nan values in 'FCF26'. We do this because we
#    will use 'FCF26' to derrive the target variable.
print(f"Dropping {unicef_df[FeatureNames.FCF26].isnull().mean() * 100:2f} % of rows corresponding to nan values in 'FCF26'.")
unicef_df = unicef_df.dropna(subset=[FeatureNames.FCF26])

# We introduce a depression_map that maps categories in the "FCF26" feature to
# binary values; where 0 = not depressed, and 1 = depressed.
depression_map = {
    "NEVER": 0, # Never sad implies never depressed (0).
    "A FEW TIMES A YEAR": 0, # Few times a year could be one-off sad life events. Therefore not depressed (0).
    "MONTHLY": 0, # Few times a year could be one-off monthly events. Therefore not depressed (0).
    "WEEKLY": 1, # Weekly is consistent and short frequent. Therefore depressed (1).
    "DAILY": 1, # Daily frequency of sadness suggests depression. Therefore depressed (1).
}
# 3) Remove rows corresponding to "NO RESPONSE" on "FCF26", the feature we will
#    use to derive the target. This Removes 0.78% of total rows.
unicef_df = unicef_df[unicef_df[FeatureNames.FCF26] != "NO RESPONSE"]

# 4) Map "FCF26" to binary feature of 0 = Not depressed, and 1 = Depressed
#    based on the depression_map.
unicef_df[FeatureNames.FCF26_BINARY] = unicef_df[FeatureNames.FCF26].map(depression_map)
print(f"Target: {FeatureNames.FCF26_BINARY} has {unicef_df[FeatureNames.FCF26_BINARY].isnull().mean() * 100:2f} % nan values")

# 5) Use Cramer's V score to select features that have a sufficient correlation
#    with the target "FCF26_binary".
#    Note correlation does not imply causation, so we cannot necessarily infer
#    causality without sufficient evidence of a mechanism that causally links
#    features highly correlated with the target.
significant_features_df = perform_cramers_v(
    df=unicef_df,
    # consider p > 0.05 a significant feature. This is tunable to be more/less
    # aggressive (higher being stricter).
    min_score=0.05,
    target_column_name=FeatureNames.FCF26_BINARY,
    ignore_columns=[
        FeatureNames.FCF26_BINARY, # FCF26_binary is target so remove FCF26_binary.
        FeatureNames.FCF26,  # target is derived from FCF26 so remove FCF26.
    ]
)

# 6) Remove some features manually on further inspection.
#     - Remove ethnicity feature to prevent racial bias. We need sufficient
#       ethical justification to include ethnicity.
#     - Remove wscore feature because it highly correlated with happiness.
#       Such that it could even be considered alternate target. When removing
#       wscore we note it as an important feature for our analysis.

significant_features_df = remove_features_from_df(
    df=significant_features_df,
    features_to_remove = [
        # Remove ethnicity feature to prevent racial bias.
        FeatureNames.ETHNICITY,
        FeatureNames.WSCORE,
    ]
)
significant_features: list[str] = significant_features_df["feature_name"].to_list()

# 7) Filter the unicef dataframe, including only significant features and the
#    target "FCF26_binary".
unicef_df_filtered = unicef_df[significant_features + ["FCF26_binary"]].copy()

# 8) Apply feature config to transform features into a numerical format that
#    our model can understand. Nominal Features we one-hot encode, and drop the
#    first column in the encoding to remove redundant information.
nominal_features = [
    # HC4: Main material of the dwelling floor.
    FeatureNames.HC4,
    # HC5: Main material of the roof.
    FeatureNames.HC5,
    # HH7: Region
    FeatureNames.HH7,
    # WS1: What is the main source of drinking water used by members of your
    # household?
    FeatureNames.WS1,
    # WS11: What kind of toilet facility do members of your household usually
    # use?
    FeatureNames.WS11,
]
unicef_df_filtered = pd.get_dummies(
    unicef_df_filtered,
    columns=nominal_features,
    # We set drop_first=True to reduce redundant information, a matrix of all
    # zeroes implies the first category.
    drop_first=True,
)
# 9) For the categorical features where a natural ordering makes sense, we
#    transform them using a map, mapping the category to a numeric value our
#    model will understand.
unicef_df_filtered = apply_feature_config(df=unicef_df_filtered, config=feature_config)
unicef_df_filtered.head()
null_counts = unicef_df_filtered.isnull().sum()
print(null_counts[null_counts > 0])

null_summary = pd.DataFrame({
    "count": unicef_df_filtered.isnull().sum(),
    "pct": unicef_df_filtered.isnull().mean() * 100
})
print(null_summary[null_summary["count"] > 0].round(2))
print(unicef_df_filtered["CL3"].value_counts(dropna=False))

Dropping 0.782556 % of rows corresponding to nan values in 'FCF26'.
Target: FCF26_binary has 0.000000 % nan values
LS2       343
LS1       279
WS4      1961
LS3       279
VT22B     279
MA2      3124
VT22A     279
LS4       279
VT22C     279
FCD2H    1795
FCD2J    1795
VT20      279
VT22D     279
WB4       279
dtype: int64
       count    pct
LS2      343   2.63
LS1      279   2.14
WS4     1961  15.04
LS3      279   2.14
VT22B    279   2.14
MA2     3124  23.96
VT22A    279   2.14
LS4      279   2.14
VT22C    279   2.14
FCD2H   1795  13.77
FCD2J   1795  13.77
VT20     279   2.14
VT22D    279   2.14
WB4      279   2.14
CL3
0.0     9897
2.0      684
1.0      592
3.0      480
4.0      281
5.0      250
6.0      155
7.0       94
10.0      90
8.0       82
12.0      55
14.0      50
9.0       49
30.0      39
20.0      38
15.0      29
21.0      29
18.0      17
35.0      13
24.0      11
16.0      10
22.0      10
28.0       8
11.0       7
13.0       7
25.0       7
60.0       6
42.0       6
36.0    

In [86]:
unicef_df_filtered.columns

Index(['HH1', 'CL3', 'LS2', 'LS1', 'WS4', 'LS3', 'VT22B', 'MA2', 'VT22A',
       'CL13', 'LS4', 'VT22C', 'HH2', 'WS7', 'HC12', 'MSTATUS', 'FCD2H',
       'FCD2J', 'VT20', 'HC8', 'VT22D', 'WB4', 'FCF26_binary', 'HC4_CEMENT',
       'HC4_CERAMIC TILES', 'HC4_DUNG', 'HC4_EARTH / SAND', 'HC4_OTHER',
       'HC4_PALM / BAMBOO', 'HC4_PARQUET OR POLISHED WOOD',
       'HC4_VINYL OR ASPHALT STRIPS', 'HC4_WOOD PLANKS', 'HC5_CEMENT',
       'HC5_CERAMIC TILES', 'HC5_IRON SHEETS / METAL / TIN', 'HC5_NO ROOF',
       'HC5_OTHER', 'HC5_PALM / BAMBOO', 'HC5_ROOFING SHINGLES',
       'HC5_RUSTIC MAT', 'HC5_THATCH / PALM LEAF', 'HC5_WOOD',
       'HC5_WOOD PLANKS', 'HH7_North', 'HH7_South',
       'WS1_DUG WELL: PROTECTED WELL', 'WS1_DUG WELL: UNPROTECTED WELL',
       'WS1_OTHER', 'WS1_PACKAGED WATER: BOTTLED WATER',
       'WS1_PIPED WATER: PIPED INTO DWELLING',
       'WS1_PIPED WATER: PIPED TO NEIGHBOUR',
       'WS1_PIPED WATER: PIPED TO YARD / PLOT',
       'WS1_PIPED WATER: PUBLIC TAP / STANDPI